# Long-Running & Asynchronous Agents

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the transition from a fragile polling script to a robust, durable, event-driven agent architecture.

We will cover 4 distinct patterns:
1. **The Polling Anti-Pattern:** Wasting CPU cycles waiting for a flag.
2. **State Serialization (Checkpointing):** Freezing the agent to a database.
3. **Idempotent Webhook Wakeup:** Surviving duplicate network events safely.
4. **Stale State Revalidation:** Catching a world-state change after a long delay.

---
## Pattern 1: The Polling Anti-Pattern

Here, the script stays alive, burning compute and risking complete memory loss if the server crashes while sleeping.

In [ ]:
def polling_agent_anti_pattern():
    print("[Agent] Proposing refund. Waiting for approval...")
    
    # Simulating a database flag
    human_approved = False
    attempts = 0
    
    while not human_approved and attempts < 3:
        print(f"[Agent] Sleeping... (attempt {attempts+1})")
        # time.sleep(60) # Commented out so notebook runs fast
        attempts += 1
        
    print("🚨 [Danger] If the server crashed during those sleeps, the agent memory is gone forever.")

polling_agent_anti_pattern()


---
## Pattern 2: State Serialization (Durable Execution)

Instead of sleeping, the agent serializes its exact context to a Mock Database and terminates the Python process.

In [ ]:
class MockDB:
    def __init__(self):
        self.store = {}
        
    def save(self, job_id, state):
        self.store[job_id] = state
        print(f"\n[DB] Saved State for Job {job_id}")
        
    def load(self, job_id):
        return self.store.get(job_id)

db = MockDB()

def checkpoint_and_kill(job_id, agent_memory):
    print("[Agent] Proposing refund. I will now checkpoint my state and terminate.")
    
    state_to_save = {
        "status": "WAITING_FOR_APPROVAL",
        "proposed_action": "refund_user",
        "memory": agent_memory,
        "timestamp": time.time()
    }
    
    db.save(job_id, state_to_save)
    print("💀 [System] Agent process terminated successfully. Zero compute used.")

checkpoint_and_kill("job_123", ["User complained", "I proposed a $50 refund"])


---
## Pattern 3: Idempotent Webhook Wakeup

The human clicks 'Approve', firing a webhook. But what if the webhook fires twice? The agent must use Idempotency Keys to prevent double-charging.

In [ ]:
executed_actions = set() # Simulating API backend

def target_api_refund(idempotency_key):
    if idempotency_key in executed_actions:
        return "200 OK (Cached Success - Duplicate Prevented)"
    executed_actions.add(idempotency_key)
    return "200 OK (Refund Processed)"

def webhook_wakeup(job_id):
    print(f"\n[Webhook] Received approval for Job {job_id}. Waking agent...")
    
    state = db.load(job_id)
    if not state or state["status"] != "WAITING_FOR_APPROVAL":
        print("❌ [Agent] Invalid job state.")
        return
        
    print(f"[Agent] Resumed. Memory restored: {state['memory']}")
    
    idempotency_key = f"refund_{job_id}"
    response = target_api_refund(idempotency_key)
    print(f"[Agent] Executed Action. API Response: {response}")

print("--- Firing Webhook (First Time) ---")
webhook_wakeup("job_123")

print("--- Firing Webhook AGAIN (Network Retry) ---")
webhook_wakeup("job_123")


---
## Pattern 4: Stale State Revalidation

If a human waits 14 days to click approve, the world state has changed. The agent must re-verify assumptions before executing the stale proposal.

In [ ]:
def webhook_wakeup_with_revalidation(job_id, current_world_state):
    print(f"\n[Webhook] Received delayed approval for Job {job_id}.")
    state = db.load(job_id)
    
    proposal_age = time.time() - state["timestamp"]
    
    print("[Agent] Re-validating world state before executing...")
    if current_world_state["user_status"] == "deleted":
        print("🚨 [Agent] ABORT. The user was deleted while I was sleeping. The proposal is stale.")
        return False
        
    print("✅ [Agent] State is valid. Executing...")
    return True

# We re-save the job
db.save("job_999", {"status": "WAITING_FOR_APPROVAL", "timestamp": time.time() - 86400}) # 1 day old

# The user was deleted by another admin during the 1-day wait
world_state = {"user_status": "deleted"} 

webhook_wakeup_with_revalidation("job_999", world_state)
